# sum-and-broadcast-duality — ex3: mean_back: derive from sum_back by dividing by the reduction count

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `sum-and-broadcast-duality`. Running the final beacon cell reports progress against the `Backprop: sum/broadcast duality` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """Minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries optional `.recipe`,
    `.requires_grad`, and `.grad` (the accumulated gradient at leaves)."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
        self.grad = None
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Backprop: sum/broadcast duality` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`sum-and-broadcast-duality`** (exercise 3). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "sum-and-broadcast-duality"
DD_SUBTOPIC = "Backprop: sum/broadcast duality"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## `mean_back` — mean is `sum / n`, so its backward is `sum_back / n`

Ex1 wrote `sum_back` (broadcast back across the reduced axis) and `broadcast_back` (sum out expanded axes) as duals. Ex2 verified the adjoint identity for sum. The third facet derives a NEW back fn from ex1's sum_back: mean is just sum divided by the reduction count, so its backward is the same broadcast-back divided by the same count.

```python
# Forward: out = x.mean(dim=k) == x.sum(dim=k) / x.shape[k]
# Backward:
#   d/dx [sum(x, k) / N] = sum_back(grad_out, x, k) / N
def mean_back(grad_out, out, x, dim, keepdim=False):
    n = x.shape[dim]
    return sum_back(grad_out, out, x, dim, keepdim) / n
```

**Why this works without re-deriving from scratch.** The chain rule applied to a scalar multiple is dead simple: `d(c*f)/dx = c * df/dx`. Mean is `(1/N) * sum`. So its gradient is `(1/N) * grad-of-sum`. We're not approximating — this is the exact derivative.

**The reduction count is `x.shape[dim]`, not `out.numel()`.** For a multi-axis mean `x.mean(dim=(0,1))`, the count is `x.shape[0] * x.shape[1]`. The single-axis case used here is just `x.shape[dim]` — the size of the axis we collapsed.

**Why this is the duality showing up in derived form.** Mean is a linear map with a scalar normalization. Sum is a linear map without it. The duality (`sum_back` broadcasts) transfers through the scalar untouched — `mean_back` is the same broadcast, scaled. It's how every loss function with `.mean()` gets its gradient: same machinery as `sum_back`, just divided.

### Exercise 3 — mean_back: derive from sum_back by dividing by the reduction count

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Apply
> LO: Apply the chain rule's scalar-multiple invariance to derive mean_back from sum_back: since mean(x, dim) = sum(x, dim) / N, the backward is the same broadcast-back divided by the same N.
> Keywords: mean, back-fn, sum-back, scalar-multiple, chain-rule
> ```

**KCs targeted:** `mean-is-scaled-sum`, `scalar-factor-passes-through-backward`

Implement two functions, building on ex1:

**1. `ex3_sum_back(grad_out, out, x, dim, keepdim=False)`** — same as ex1. Re-insert axis via `unsqueeze` if `keepdim=False`, then `.expand_as(x).clone()`.

**2. `ex3_mean_back(grad_out, out, x, dim, keepdim=False)`** — backward for `out = x.mean(dim=dim, keepdim=keepdim)`. Reuse `ex3_sum_back` and divide the result by `x.shape[dim]` (the reduction count):

```python
def mean_back(grad_out, out, x, dim, keepdim=False):
    n = x.shape[dim]
    return sum_back(grad_out, out, x, dim, keepdim) / n
```

Inputs (both functions):
- `grad_out`: `torch.Tensor`, gradient flowing in from the next node — shape matches `out`.
- `out`: `torch.Tensor`, the FORWARD output (kept for ABI consistency with PyTorch back-fns — `mean_back` doesn't actually use it).
- `x`: `torch.Tensor`, the input the forward op reduced over.
- `dim`: int, axis index.
- `keepdim`: bool, whether the forward kept the axis as size 1.

Constraints:
- `mean_back` MUST reuse `sum_back` (don't reimplement the broadcasting from scratch).
- Output shape must equal `x.shape`.
- Output values must match `torch.autograd` on the equivalent forward.

In [ ]:
def ex3_sum_back(grad_out, out, x, dim, keepdim=False):
    if not keepdim:
        grad_out = grad_out.unsqueeze(dim)
    return grad_out.expand_as(x).clone()


def ex3_mean_back(grad_out, out, x, dim, keepdim=False):
    # Mean = Sum / N → its derivative is sum_back / N (chain rule on a scalar mult).
    n = x.shape[dim]
    return ex3_sum_back(grad_out, out, x, dim, keepdim) / n


<details><summary>Solution</summary>

```python
def ex3_sum_back(grad_out, out, x, dim, keepdim=False):
    if not keepdim:
        grad_out = grad_out.unsqueeze(dim)
    return grad_out.expand_as(x).clone()


def ex3_mean_back(grad_out, out, x, dim, keepdim=False):
    # Mean = Sum / N → its derivative is sum_back / N (chain rule on a scalar mult).
    n = x.shape[dim]
    return ex3_sum_back(grad_out, out, x, dim, keepdim) / n
```

**Mean is sum scaled.** `x.mean(dim=k) = x.sum(dim=k) / x.shape[k]`. Apply the chain rule: `d/dx[c * f(x)] = c * df/dx`. The scalar `1/N` passes through the linear backward unchanged — so `mean_back = sum_back / N`. No new derivation needed; the duality from ex1 transfers via scalar multiplication.

**Why `x.shape[dim]`, not `out.numel()`.** For a single-axis mean, these are equal. For a multi-axis mean (`x.mean(dim=(0,1))`), the count is the product of the reduced axes' sizes. The drill keeps to single-axis for clarity — multi-axis is a straightforward extension once the principle is in.

**Why `out` is in the signature but unused.** PyTorch's back-fn ABI passes `out` (the forward output) for back-fns that need it (e.g. `sigmoid_back` reuses `sigmoid(x)` from forward). `mean_back` doesn't need it, but keeping the parameter in the signature lets the dispatch table store all back-fns with one ABI.

**The adjoint identity still holds.** Same `<Ax, y> = <x, A^T y>` as ex2 — just with `A = mean(dim)` and `A^T = mean_back`. Verifying this is the gold-standard test for any back-fn; the scalar `1/N` factor doesn't change the identity, only the magnitude on both sides.

**Where this shows up in real models.** Every cross-entropy loss that averages over a batch uses `mean_back` on the reverse pass. Every BatchNorm running-mean update is a `mean` forward whose backward (if it weren't intentionally detached) would be `mean_back`. Same machinery as `sum_back`, just divided.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex3'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex3',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()